In [7]:
!pip install ultralytics

In [2]:
import os
import json
import shutil
import yaml
from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
DRIVE_DIR = "/content/drive/MyDrive/ParkFlow_AI"
DRIVE_ZIP_PATH = os.path.join(DRIVE_DIR, "yolo_dataset_processed.zip")

LOCAL_RAW = Path("/content/raw_data")
LOCAL_YOLO = Path("/content/yolo_dataset")

os.makedirs(DRIVE_DIR, exist_ok=True)

# 3. Kaggle Credentials (Replace with your actual keys)
os.environ['KAGGLE_USERNAME'] = "sandakannipunajith"
os.environ['KAGGLE_KEY'] = "KGAT_8eb82a391617f56c3c05696cdb9f87fc"

Mounted at /content/drive


In [3]:
def coco_bbox_to_yolo(bbox, img_w, img_h):
    """Your provided logic for normalized [xc, yc, w, h]."""
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_w
    y_center = (y_min + h / 2) / img_h
    return x_center, y_center, w / img_w, h / img_h

def convert_split(json_path, src_image_dir, out_image_dir, out_label_dir, category_id_to_yolo):
    """Your provided logic to process COCO files into YOLO labels."""
    with open(json_path) as f:
        data = json.load(f)

    images = {img["id"]: img for img in data["images"]}
    ann_by_image = {img_id: [] for img_id in images}
    for ann in data["annotations"]:
        if not ann.get("iscrowd", 0):
            ann_by_image[ann["image_id"]].append(ann)

    out_image_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok=True)

    for img_id, img_info in images.items():
        filename = img_info["file_name"]
        stem = Path(filename).stem

        # High-speed local copy
        shutil.copy2(src_image_dir / filename, out_image_dir / filename)

        lines = []
        for ann in ann_by_image.get(img_id, []):
            cat_id = ann["category_id"]
            if cat_id not in category_id_to_yolo: continue

            yolo_class = category_id_to_yolo[cat_id]
            xc, yc, nw, nh = coco_bbox_to_yolo(ann["bbox"], img_info["width"], img_info["height"])

            # Clamp to [0, 1] as per your provided logic
            xc, yc, nw, nh = [max(0.0, min(1.0, v)) for v in [xc, yc, nw, nh]]
            lines.append(f"{yolo_class} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")

        with open(out_label_dir / f"{stem}.txt", "w") as f:
            f.write("\n".join(lines))

In [4]:
if os.path.exists(DRIVE_ZIP_PATH):
    print("🚀 Restoring processed dataset from Drive...")
    shutil.copy2(DRIVE_ZIP_PATH, "/content/yolo_dataset.zip")
    !unzip -q /content/yolo_dataset.zip -d /content/
else:
    print("📂 Starting raw conversion...")
    !kaggle datasets download -d ammarnassanalhajali/pklot-dataset -p {LOCAL_RAW}
    !unzip -q {LOCAL_RAW}/pklot-dataset.zip -d {LOCAL_RAW}

    # As per your dataset structure: 0:spaces, 1:space-empty, 2:space-occupied
    category_map = {0: 0, 1: 1, 2: 2}

    for split in ["train", "valid"]:
        convert_split(
            LOCAL_RAW / split / "_annotations.coco.json",
            LOCAL_RAW / split,
            LOCAL_YOLO / split / "images",
            LOCAL_YOLO / split / "labels",
            category_map
        )

    # Generate data.yaml with your exact 3 classes
    data_yaml = {
        'path': str(LOCAL_YOLO),
        'train': 'train/images',
        'val': 'valid/images',
        'nc': 3,
        'names': ['spaces', 'space-empty', 'space-occupied']
    }
    with open(LOCAL_YOLO / "data.yaml", "w") as f:
        yaml.dump(data_yaml, f)

    # Backup to Drive
    shutil.make_archive("/content/yolo_dataset", 'zip', LOCAL_YOLO)
    shutil.copy2("/content/yolo_dataset.zip", DRIVE_ZIP_PATH)
    print("✅ Conversion and Drive backup complete.")

📂 Starting raw conversion...
Dataset URL: https://www.kaggle.com/datasets/ammarnassanalhajali/pklot-dataset
License(s): unknown
 96% 809M/843M [00:03<00:00, 142MB/s]
100% 843M/843M [00:04<00:00, 218MB/s]
✅ Conversion and Drive backup complete.


In [10]:
# Use yolo11n as the base for high-speed inference on your MacBook Air M4 later
from ultralytics import YOLO
model = YOLO("yolo26n.pt")

model.train(
    data=os.path.join(LOCAL_YOLO, "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    project=f"{DRIVE_DIR}/training_results",
    name="parkflow_3class_run",
    device=0
)

print(f"✅ Training finished. Download your weights from: {DRIVE_DIR}/training_results")

Ultralytics 8.4.17 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=parkflow_3class_run2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10